In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/small_bench/checkpoints/pre_cell_1.pickle

In [ ]:
%%PandasProfile
### cell 1 ###

# Optimized Code
# Pre-bind functions/attributes for faster lookups
def load_survey_data(file_path, year, skip):
    df = _read_csv(file_path, low_memory=False, encoding='ISO-8859-1', skiprows=skip)
    out_file = f"kaggle_survey_{year}.csv"
    if not _exists(out_file):
        _to_csv(df, out_file, index=False)
    return df

# Constants
factor = 1
_read_csv = pd.read_csv
_exists = os.path.exists
_to_csv = pd.DataFrame.to_csv

# Create only the needed chart subdirectories (parents are created automatically)
root = (
    '/home/dias-benchmarks/notebooks/paultimothymooney/'
    'kaggle-survey-2022-all-results/kaggle/working/individual_charts'
)
for sub in ('data', 'charts'):
    os.makedirs(f'{root}/{sub}', exist_ok=True)

# Load and sample survey data for years 2018–2022
file_info = {
    2018: ('multipleChoiceResponses.csv', 1),
    2019: ('multiple_choice_responses.csv', 1),
    2020: ('kaggle_survey_2020_responses.csv', 1),
    2021: ('kaggle_survey_2021_responses.csv', 1),
    2022: ('kaggle_survey_2022_responses.csv', 1),
}
base_prefix = (
    '/home/dias-benchmarks/notebooks/paultimothymooney/'
    'kaggle-survey-2022-all-results/input/kaggle-survey-'
)

for year, (fname, skip) in file_info.items():
    path = f"{base_prefix}{year}/{fname}"
    globals()[f'responses_df_{year}'] = (
        load_survey_data(path, year, skip)
        .sample(frac=factor, random_state=0)
    )
